In [1]:
"""
A/B Testing Statistical Analysis for Customer Retention Experiment
================================================================================
Database: ABTesting (PostgreSQL)
Problem: Evaluate whether the Loyalty Points & Public Recognition Program (Treatment)
         causally increases 90-Day Client Retention compared to Standard Onboarding (Control).
In this experiment evaluating loyalty points and public recognition program,
the hypotheses are structured around the primary metric: 90-Day 
Client Retention Rate.
    Let:Pc = True 90-day retention rate for the Control Group (Standard Onboarding)
    Pt= True 90-day retention rate for the Treatment Group (Loyalty Points & Recognition)
1.Null Hypothesis (H_0)Definition: The status quo assumption that the intervention has no positive effect.
Any observed improvement in retention is purely due to random chance.
    Business Meaning: The loyalty program does not increase client retention. If we roll it out,
    any gains we see in our sample are accidental noise.
2. Alternative Hypothesis (H_1)Definition: The experimental assumption that the intervention successfully
causes a meaningful, statistically significant increase in client retention.
    Business Meaning: The loyalty program genuinely improves client retention. The lift observed in
    the Treatment group is real and caused by the rewards program rather than random chance.
"""

import numpy as np
import pandas as pd
import psycopg2, math
from scipy import stats
import statsmodels.stats.api as sms

# ==============================================================================
# DATABASE CONFIGURATION & CONNECTION
# ==============================================================================
DB_CONFIG = {
    "dbname": "ABTesting",
    "user": "postgres",
    "password": "your_password",  # Update with your database password
    "host": "localhost",
    "port": "5432",
}


def get_db_connection():
    """Establish connection to the PostgreSQL database."""
    try:
        conn = psycopg2.connect(**DB_CONFIG)
        print(" Successfully connected to PostgreSQL database: ABTesting")
        return conn
    except Exception as e:
        print(f" Database connection failed: {e}")
        raise

In [2]:
conn = get_db_connection()

# ==============================================================================
# STEP 1: RESOLVING SELECTION BIAS & CONFOUNDING NOISE (DATA EXTRACTION)
# ==============================================================================
def extract_experiment_data(conn):
    """
    Extracts raw client-level records from PostgreSQL.
    
    How this resolves the core problem:
    By querying experiment_assignments, we ensure that clients are analyzed 
    strictly according to the randomized group they were assigned to (Intent-to-Treat),
    eliminating selection bias.
    """
    query = """
    SELECT 
        c.client_id,
        ea.variant,
        ca.is_active_90d AS retained_90d,
        ca.total_revenue,
        ca.milestones_completed
    FROM clients c
    JOIN experiment_assignments ea ON c.client_id = ea.client_id
    JOIN client_activity ca ON c.client_id = ca.client_id
    WHERE ea.experiment_name = 'loyalty_program_v1';
    """
    print("\n--- STEP 1: Fetching randomized experiment data from database ---")
    df = pd.read_sql_query(query, conn)
    print(f"Extracted {len(df):,} total records from database.")
    return df


df = extract_experiment_data(conn)
df.head(10)

 Successfully connected to PostgreSQL database: ABTesting

--- STEP 1: Fetching randomized experiment data from database ---


/tmp/ipykernel_9506/3470592870.py:28: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)


Extracted 200,000 total records from database.


,client_id,variant,retained_90d,total_revenue,milestones_completed
0,CLI-000001,Treatment,0,4927.95,1
1,CLI-000004,Treatment,1,6638.64,7
2,CLI-000005,Control,1,9380.12,9
3,CLI-000006,Treatment,1,9320.23,3
4,CLI-000027,Treatment,0,3750.80,7
5,CLI-000029,Treatment,0,5838.50,5
6,CLI-000031,Treatment,0,2741.14,1
7,CLI-000032,Control,0,9226.64,10
8,CLI-000036,Treatment,0,2648.54,3
9,CLI-000039,Treatment,0,9451.15,4


In [3]:
# ==============================================================================
# STEP 2: RESOLVING RANDOMNESS & SAMPLING ERROR (SAMPLE SIZE & POWER CHECK)
# ==============================================================================
def verify_sample_size(p1=0.40, mde=0.06, alpha=0.05, power=0.80):
    """
    Calculates the minimum required sample size per group before drawing conclusions.
    
    How this resolves the core problem:
    Prevents Type I errors (false positives) and Type II errors (false negatives)
    by confirming whether the sample size in the database is sufficiently powered.

    Alpha is the significance level: the maximum acceptable probability of a 
    Type I error—rejecting the null hypothesis when it is true.

    The calculated sample size (1,068 clients per group, or 2,136 total) is
    the minimum amount of data you need to collect to ensure your statistical
    test has enough power (80%) to detect your target Minimum Detectable Effect
    without falling victim to false negatives.
    """
    p2 = p1 + mde
    effect_size = sms.proportion_effectsize(p1, p2)
    required_n = sms.NormalIndPower().solve_power(
        effect_size=effect_size, power=power, alpha=alpha, ratio=1.0
    )
    required_n = math.ceil(required_n)
    
    print("\n--- STEP 2: Pre-Experiment Sample Size & Power Verification ---")
    print(f"Baseline Retention (Control): {p1:.1%}")
    print(f"Target Minimum Detectable Effect (MDE): +{mde:.1%}")
    print(f"Required Sample Size per Group: {required_n:,} clients")
    return required_n


# Step 2: Validate sample size requirements
required_n = verify_sample_size()
actual_n = df["variant"].value_counts().min()
if actual_n < required_n:
    print(f" Caution: Actual sample size ({actual_n:,}) is below required power threshold ({required_n:,}).")


--- STEP 2: Pre-Experiment Sample Size & Power Verification ---
Baseline Retention (Control): 40.0%
Target Minimum Detectable Effect (MDE): +6.0%
Required Sample Size per Group: 1,068 clients


In [4]:
# ==============================================================================
# STEP 3: RESOLVING DATA INTEGRITY & RANDOMIZATION FAILURE (SRM CHECK)
# ==============================================================================
def check_sample_ratio_mismatch(df):
    """
    Performs a Chi-Square Goodness-of-Fit test to verify 50/50 randomization.

    Remember we split the data set into treatment and control groups. We need
    to check that the split was 50/50 and not say for example 60/40. Supposing
    the data has been split badly then the experiment would not work well.
    """
    counts = df["variant"].value_counts()
    n_control = counts.get("Control", 0)
    n_treatment = counts.get("Treatment", 0)
    total = n_control + n_treatment

    # Expected count under ideal 50/50 split
    expected = [total / 2, total / 2]
    observed = [n_control, n_treatment]

    _, p_value_srm = stats.chisquare(f_obs=observed, f_exp=expected)

    print("\n--- STEP 3: Sample Ratio Mismatch (SRM) Integrity Check ---")
    print(f"Observed - Control: {n_control:,} | Treatment: {n_treatment:,}")
    print(f"SRM Test p-value: {p_value_srm:.4f}")

    if p_value_srm < 0.01:
        raise ValueError(" CRITICAL WARNING: Sample Ratio Mismatch detected! Check routing assignment.")
    else:
        print(" Randomization integrity verified (No SRM detected).")


# Step 3: Run data integrity check
check_sample_ratio_mismatch(df)



--- STEP 3: Sample Ratio Mismatch (SRM) Integrity Check ---
Observed - Control: 99,926 | Treatment: 100,074
SRM Test p-value: 0.7407
 Randomization integrity verified (No SRM detected).


In [5]:
# ==============================================================================
# STEP 4: RESOLVING UNCERTAINTY IN PRIMARY METRIC (TWO-PROPORTION Z-TEST)
# ==============================================================================
def evaluate_primary_metric(df, alpha=0.05):
    """
    Executes a two-sided Z-test for two proportions on 90-Day Retention Rate.
    
    How this resolves the core problem:
    Quantifies the exact probability (p-value) that the observed lift in retention
    happened purely by random chance vs. a real causal impact of the loyalty program.
    """
    summary = df.groupby("variant")["retained_90d"].agg(["sum", "count", "mean"])
    
    x_control = summary.loc["Control", "sum"]
    n_control = summary.loc["Control", "count"]
    p_control = summary.loc["Control", "mean"]

    x_treatment = summary.loc["Treatment", "sum"]
    n_treatment = summary.loc["Treatment", "count"]
    p_treatment = summary.loc["Treatment", "mean"]

    # Calculate Z-statistic and p-value
    z_stat, p_value = sms.proportions_ztest(
        [x_treatment, x_control], [n_treatment, n_control], alternative="two-sided"
    )

    # Calculate Confidence Interval for difference in proportions
    diff = p_treatment - p_control
    se_diff = math.sqrt(
        (p_control * (1 - p_control) / n_control) + (p_treatment * (1 - p_treatment) / n_treatment)
    )
    ci_lower = diff - (1.96 * se_diff)
    ci_upper = diff + (1.96 * se_diff)

    print("\n--- STEP 4: Primary Metric Evaluation (90-Day Retention Rate) ---")
    print(f"Control Retention Rate:   {p_control:.2%}")
    print(f"Treatment Retention Rate: {p_treatment:.2%}")
    print(f"Observed Absolute Lift:   {diff:+.2%} (95% CI: [{ci_lower:+.2%}, {ci_upper:+.2%}])")
    print(f"Z-Score:                  {z_stat:.4f}")
    print(f"p-value:                  {p_value:.6e}")

    is_significant = p_value < alpha
    return is_significant, diff, p_value


# Step 4: Evaluate primary metric
is_significant, diff, p_value = evaluate_primary_metric(df)
print("The test is significant:", is_significant)
print(diff)
print(p_value)


--- STEP 4: Primary Metric Evaluation (90-Day Retention Rate) ---
Control Retention Rate:   39.73%
Treatment Retention Rate: 46.98%
Observed Absolute Lift:   +7.25% (95% CI: [+6.82%, +7.68%])
Z-Score:                  32.7121
p-value:                  1.052629e-234
The test is significant: True
0.07249832650008364
1.0526286879057852e-234


In [6]:
# ==============================================================================
# STEP 5: RESOLVING UNINTENDED BUSINESS HARM (GUARDRAIL METRIC EVALUATION)
# ==============================================================================
def evaluate_guardrail_metrics(df, alpha=0.05):
    """
    Evaluates Average Revenue Per Client (ARPC) using Welch's t-test.
    Did the loyalty program improve retention without damaging revenue per client?
    
    How this resolves the core problem:
    Ensures that the new loyalty feature does not harm revenue per user (e.g., if clients 
    stay active longer but spend less overall due to program incentives).
    """
    control_rev = df[df["variant"] == "Control"]["total_revenue"]
    treatment_rev = df[df["variant"] == "Treatment"]["total_revenue"]

    mean_c = control_rev.mean()
    mean_t = treatment_rev.mean()

    # Welch's t-test for unequal variances
    t_stat, p_value_rev = stats.ttest_ind(treatment_rev, control_rev, equal_var=False)

    print("\n--- STEP 5: Guardrail Metric Check (Average Revenue Per Client) ---")
    print(f"Control Mean Revenue:   ${mean_c:,.2f}")
    print(f"Treatment Mean Revenue: ${mean_t:,.2f}")
    print(f"Revenue p-value:        {p_value_rev:.4f}")

    if p_value_rev >= alpha or mean_t >= mean_c:
        print(" Guardrail check passed: Revenue per account is maintained or improved.")
    else:
        print(" WARNING: Treatment group showed a statistically significant revenue decline!")


evaluate_guardrail_metrics(df)

# Step 6: Final decision framework
print("\n==============================================================================")
print("FINAL BUSINESS DECISION FRAMEWORK")
print("==============================================================================")
if is_significant and diff > 0:
    print(" STATISTICAL DECISION: Reject Null Hypothesis (H0).")
    print(" ACTIONABLE RECOMMENDATION: Roll out the Loyalty Points & Recognition Program")
    print("                             to 100% of Infortrace consulting clients.")
else:
    print(" STATISTICAL DECISION: Fail to Reject Null Hypothesis (H0).")
    print(" ACTIONABLE RECOMMENDATION: Do NOT roll out the treatment. The loyalty program")
    print("                             did not demonstrate a statistically significant lift.")
print("==============================================================================\n")


--- STEP 5: Guardrail Metric Check (Average Revenue Per Client) ---
Control Mean Revenue:   $5,516.18
Treatment Mean Revenue: $5,502.99
Revenue p-value:        0.2553
 Guardrail check passed: Revenue per account is maintained or improved.

FINAL BUSINESS DECISION FRAMEWORK
 STATISTICAL DECISION: Reject Null Hypothesis (H0).
 ACTIONABLE RECOMMENDATION: Roll out the Loyalty Points & Recognition Program
                             to 100% of Infortrace consulting clients.

